# Feature Engineering for Anime

## Imports and Set Ups

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import random
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_columns', None)

# Configuration
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = Path("/content/drive/MyDrive/yuran_files/datasets_needed")   # change if needed
ANIME_COMPLETE_PATH = BASE_PATH / "anime_complete.csv"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Data Profiling of anime_complete.csv

In [ ]:
# Access anime_complete.csv
anime_complete = pd.read_csv(ANIME_COMPLETE_PATH)

# Rename MAL_ID to anime_id
anime_complete.rename(columns={'MAL_ID': 'anime_id'}, inplace=True)

In [ ]:
# View anime_complete
print("anime_complete")
anime_complete.head()

anime_complete


,anime_id,Name,Score,Genres,English name,Japanese name,Type,Episodes,Aired,Premiered,Producers,Licensors,Studios,Source,Duration,Rating,Ranked,Popularity,Members,Favorites,Watching,Completed,On-Hold,Dropped,Plan to Watch,Score-10,Score-9,Score-8,Score-7,Score-6,Score-5,Score-4,Score-3,Score-2,Score-1,Characters,Staff,synopsis
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",Cowboy Bebop,カウボーイビバップ,TV,26,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,Bandai Visual,"Funimation, Bandai Entertainment",Sunrise,Original,24 min. per ep.,R - 17+ (violence & profanity),28.0,39,1251960,61971,105808,718161,71513,26678,329800,229170.0,182126.0,131625.0,62330.0,20688.0,8904.0,3184.0,1357.0,741.0,1580.0,"Bartender B, Bonnaro, Stella, Long, Ping, Ein,...","Alonso, Tasio, Satou, Dai, Stanzione, Carol, N...","In the year 2071, humanity has colonized sever..."
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space",Cowboy Bebop:The Movie,カウボーイビバップ 天国の扉,Movie,1,"Sep 1, 2001",Unknown,"Sunrise, Bandai Visual",Sony Pictures Entertainment,Bones,Original,1 hr. 55 min.,R - 17+ (violence & profanity),159.0,518,273145,1174,4143,208333,1935,770,57964,30043.0,49201.0,49505.0,22632.0,5805.0,1877.0,577.0,221.0,109.0,379.0,"Volaju, Vincent, Robber, Rather, Mark, Spy C, ...","Shibata, Hidekatsu, Isobe, Tsutomu, Kramer, St...","other day, another bounty—such is the life of ..."
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen",Trigun,トライガン,TV,26,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,Victor Entertainment,"Funimation, Geneon Entertainment USA",Madhouse,Manga,24 min. per ep.,PG-13 - Teens 13 or older,266.0,201,558913,12944,29113,343492,25465,13925,146918,50229.0,75651.0,86142.0,49432.0,15376.0,5838.0,1965.0,664.0,316.0,533.0,"Sandy, Bluesummers, Legato, Schezar, Cliff, Bo...","Pappas, Constantin, Stanzione, Carol, Motoi, E...","Vash the Stampede is the man with a $$60,000,0..."
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",Witch Hunter Robin,Witch Hunter ROBIN (ウイッチハンターロビン),TV,26,"Jul 2, 2002 to Dec 24, 2002",Summer 2002,"TV Tokyo, Bandai Visual, Dentsu, Victor Entert...","Funimation, Bandai Entertainment",Sunrise,Original,25 min. per ep.,PG-13 - Teens 13 or older,2481.0,1467,94683,587,4300,46165,5121,5378,33719,2182.0,4806.0,10128.0,11618.0,5709.0,2920.0,1083.0,353.0,164.0,131.0,"Amon, Kousaka, Shintarou, Sena, Robin, Sakaki,...","Meyer, Dirk, Cericola, Dania, Takahashi, Kumik...",ches are individuals with special powers like ...
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",Beet the Vandel Buster,冒険王ビィト,TV,52,"Sep 30, 2004 to Sep 29, 2005",Fall 2004,"TV Tokyo, Dentsu",Unknown,Toei Animation,Manga,23 min. per ep.,PG - Children,3710.0,4369,13224,18,642,7314,766,1108,3394,312.0,529.0,1242.0,1713.0,1068.0,634.0,265.0,83.0,50.0,27.0,"Kissu, Slade, Beltoze, Grunide, Poala, Beet, S...","Clinkenbeard, Colleen, Hisakawa, Aya, Yamamuro...",It is the dark century and the people are suff...


In [ ]:
print("anime_complete describe()")
anime_complete.describe()

anime_complete describe()


,anime_id,Popularity,Members,Favorites,Watching,Completed,On-Hold,Dropped,Plan to Watch
count,17562.000000,17562.000000,1.756200e+04,17562.000000,17562.000000,1.756200e+04,17562.000000,17562.000000,17562.000000
mean,21477.192347,8763.452340,3.465854e+04,457.746270,2231.487758,2.209557e+04,955.049653,1176.599533,8199.831227
std,14900.093170,5059.327278,1.252821e+05,4063.473313,14046.688133,9.100919e+04,4275.675096,4740.348653,23777.691963
min,1.000000,0.000000,1.000000e+00,0.000000,0.000000,0.000000e+00,0.000000,0.000000,1.000000
25%,5953.500000,4383.500000,3.360000e+02,0.000000,13.000000,1.110000e+02,6.000000,37.000000,112.000000
50%,22820.000000,8762.500000,2.065000e+03,3.000000,73.000000,8.175000e+02,45.000000,77.000000,752.500000
75%,35624.750000,13145.000000,1.322325e+04,31.000000,522.000000,6.478000e+03,291.750000,271.000000,4135.500000
max,48492.000000,17565.000000,2.589552e+06,183914.000000,887333.000000,2.182587e+06,187919.000000,174710.000000,425531.000000


In [ ]:
print("anime_complete info()")
anime_complete.info()

anime_complete info()
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17562 entries, 0 to 17561
Data columns (total 38 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   anime_id       17562 non-null  int64 
 1   Name           17562 non-null  object
 2   Score          17562 non-null  object
 3   Genres         17562 non-null  object
 4   English name   17562 non-null  object
 5   Japanese name  17562 non-null  object
 6   Type           17562 non-null  object
 7   Episodes       17562 non-null  object
 8   Aired          17562 non-null  object
 9   Premiered      17562 non-null  object
 10  Producers      17562 non-null  object
 11  Licensors      17562 non-null  object
 12  Studios        17562 non-null  object
 13  Source         17562 non-null  object
 14  Duration       17562 non-null  object
 15  Rating         17562 non-null  object
 16  Ranked         17562 non-null  object
 17  Popularity     17562 non-null  int64 
 18  Memb

In [ ]:
# Fix data type for columns
columns_to_fix = ["Episodes", "Ranked", "Score-10", "Score-9", "Score-8", "Score-7", "Score-6", "Score-5", "Score-4", "Score-3", "Score-2", "Score-1"]

for col in columns_to_fix:
    anime_complete[col] = pd.to_numeric(anime_complete[col], errors="coerce")

anime_complete.dtypes

,0
anime_id,int64
Name,object
Score,object
Genres,object
English name,object
Japanese name,object
Type,object
Episodes,float64
Aired,object
Premiered,object


## One Hot Encoding for Genres

In [ ]:
# One-Hot Encoding for Genres

anime_complete['Genres_list'] = anime_complete['Genres'].fillna('').apply(lambda x: x.split(', '))

from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
genres_encoded = pd.DataFrame(
    mlb.fit_transform(anime_complete['Genres_list']),
    columns=mlb.classes_,
    index=anime_complete.index
)

anime_complete = pd.concat([anime_complete, genres_encoded], axis=1)
anime_complete.drop('Genres_list', axis=1, inplace=True)

# View anime_complete after one-hot encoding
print("anime_complete after one-hot encoding")
anime_complete.head()

anime_complete after one-hot encoding


,anime_id,Name,Score,Genres,English name,Japanese name,Type,Episodes,Aired,Premiered,Producers,Licensors,Studios,Source,Duration,Rating,Ranked,Popularity,Members,Favorites,Watching,Completed,On-Hold,Dropped,Plan to Watch,Score-10,Score-9,Score-8,Score-7,Score-6,Score-5,Score-4,Score-3,Score-2,Score-1,Characters,Staff,synopsis,Action,Adventure,Cars,Comedy,Dementia,Demons,Drama,Ecchi,Fantasy,Game,Harem,Hentai,Historical,Horror,Josei,Kids,Magic,Martial Arts,Mecha,Military,Music,Mystery,Parody,Police,Psychological,Romance,Samurai,School,Sci-Fi,Seinen,Shoujo,Shoujo Ai,Shounen,Shounen Ai,Slice of Life,Space,Sports,Super Power,Supernatural,Thriller,Unknown,Vampire,Yaoi,Yuri
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",Cowboy Bebop,カウボーイビバップ,TV,26.0,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,Bandai Visual,"Funimation, Bandai Entertainment",Sunrise,Original,24 min. per ep.,R - 17+ (violence & profanity),28.0,39,1251960,61971,105808,718161,71513,26678,329800,229170.0,182126.0,131625.0,62330.0,20688.0,8904.0,3184.0,1357.0,741.0,1580.0,"Bartender B, Bonnaro, Stella, Long, Ping, Ein,...","Alonso, Tasio, Satou, Dai, Stanzione, Carol, N...","In the year 2071, humanity has colonized sever...",1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space",Cowboy Bebop:The Movie,カウボーイビバップ 天国の扉,Movie,1.0,"Sep 1, 2001",Unknown,"Sunrise, Bandai Visual",Sony Pictures Entertainment,Bones,Original,1 hr. 55 min.,R - 17+ (violence & profanity),159.0,518,273145,1174,4143,208333,1935,770,57964,30043.0,49201.0,49505.0,22632.0,5805.0,1877.0,577.0,221.0,109.0,379.0,"Volaju, Vincent, Robber, Rather, Mark, Spy C, ...","Shibata, Hidekatsu, Isobe, Tsutomu, Kramer, St...","other day, another bounty—such is the life of ...",1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen",Trigun,トライガン,TV,26.0,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,Victor Entertainment,"Funimation, Geneon Entertainment USA",Madhouse,Manga,24 min. per ep.,PG-13 - Teens 13 or older,266.0,201,558913,12944,29113,343492,25465,13925,146918,50229.0,75651.0,86142.0,49432.0,15376.0,5838.0,1965.0,664.0,316.0,533.0,"Sandy, Bluesummers, Legato, Schezar, Cliff, Bo...","Pappas, Constantin, Stanzione, Carol, Motoi, E...","Vash the Stampede is the man with a $$60,000,0...",1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",Witch Hunter Robin,Witch Hunter ROBIN (ウイッチハンターロビン),TV,26.0,"Jul 2, 2002 to Dec 24, 2002",Summer 2002,"TV Tokyo, Bandai Visual, Dentsu, Victor Entert...","Funimation, Bandai Entertainment",Sunrise,Original,25 min. per ep.,PG-13 - Teens 13 or older,2481.0,1467,94683,587,4300,46165,5121,5378,33719,2182.0,4806.0,10128.0,11618.0,5709.0,2920.0,1083.0,353.0,164.0,131.0,"Amon, Kousaka, Shintarou, Sena, Robin, Sakaki,...","Meyer, Dirk, Cericola, Dania, Takahashi, Kumik...",ches are individuals with special powers like ...,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",Beet the Vandel Buster,冒険王ビィト,TV,52.0,"Sep 30, 2004 to Sep 29, 2005",Fall 2004,"TV Tokyo, Dentsu",Unknown,Toei Animation,Manga,23 min. per ep.,PG - Children,3710.0,4369,13224,18,642,7314,766,1108,3394,312.0,529.0,1242.0,1713.0,1068.0,634.0,265.0,83.0,50.0,27.0,"Kissu, Slade, Beltoze, Grunide, Poala, Beet, S...","Clinkenbeard, Colleen, Hisakawa, Aya, Yamamuro...",It is the dark century and the people are suff...,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0


## TF-IDF For Genres

In [ ]:
anime_complete['Genres'] = anime_complete['Genres'].fillna('')

tfidf = TfidfVectorizer(
    tokenizer=lambda x: x.split(', '),
    token_pattern=None
)

genres_tfidf = tfidf.fit_transform(anime_complete['Genres'])

In [ ]:
# For inspection
genres_tfidf_df = pd.DataFrame(
    genres_tfidf.toarray(),
    columns=tfidf.get_feature_names_out()
)

print("genres_tfidf_df:")
genres_tfidf_df.head()

genres_tfidf_df:


,action,adventure,cars,comedy,dementia,demons,drama,ecchi,fantasy,game,harem,hentai,historical,horror,josei,kids,magic,martial arts,mecha,military,music,mystery,parody,police,psychological,romance,samurai,school,sci-fi,seinen,shoujo,shoujo ai,shounen,shounen ai,slice of life,space,sports,super power,supernatural,thriller,unknown,vampire,yaoi,yuri
0,0.334820,0.371356,0.0,0.276259,0.0,0.0,0.387558,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.389405,0.0,0.0,0.0,0.000000,0.0,0.0,0.609782,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
1,0.319565,0.000000,0.0,0.000000,0.0,0.0,0.369900,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.533099,0.0,0.000000,0.0,0.0,0.0,0.0,0.371663,0.0,0.0,0.0,0.000000,0.0,0.0,0.582000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
2,0.372624,0.413285,0.0,0.307451,0.0,0.0,0.431316,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.433372,0.0,0.0,0.0,0.471144,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
3,0.269951,0.000000,0.0,0.000000,0.0,0.0,0.312471,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.407674,0.0,0.0,0.0,0.0,0.450332,0.0,0.566259,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.373954,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.457134,0.0,0.000000,0.0,0.0,0.000000,0.0,0.43985,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.521131,0.0,0.0,0.000000,0.0,0.0,0.570949,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Find the cosine-similiarity

cosine_sim = cosine_similarity(genres_tfidf)

In [ ]:
# Create an index mapping
indices = pd.Series(anime_complete.index, index=anime_complete['English name']).drop_duplicates()

## Normalization for Episodes/Scores

In [ ]:
columns_to_scale = ["Episodes", "Score-10", "Score-9", "Score-8", "Score-7", "Score-6", "Score-5", "Score-4", "Score-3", "Score-2", "Score-1"]

for col in columns_to_scale:
    X = np.array(anime_complete[col]).reshape(-1, 1)
    scaler = StandardScaler()
    scaler.fit(X)
    X_scaled = scaler.transform(X)
    anime_complete[f"{col}_scaled"] = X_scaled.reshape(1, -1)[0]
    anime_complete.drop(col, axis=1, inplace=True)

print("anime_complete after scaling:")
anime_complete.head()

anime_complete after scaling:


,anime_id,Name,Score,Genres,English name,Japanese name,Type,Aired,Premiered,Producers,Licensors,Studios,Source,Duration,Rating,Ranked,Popularity,Members,Favorites,Watching,Completed,On-Hold,Dropped,Plan to Watch,Characters,Staff,synopsis,Action,Adventure,Cars,Comedy,Dementia,Demons,Drama,Ecchi,Fantasy,Game,Harem,Hentai,Historical,Horror,Josei,Kids,Magic,Martial Arts,Mecha,Military,Music,Mystery,Parody,Police,Psychological,Romance,Samurai,School,Sci-Fi,Seinen,Shoujo,Shoujo Ai,Shounen,Shounen Ai,Slice of Life,Space,Sports,Super Power,Supernatural,Thriller,Unknown,Vampire,Yaoi,Yuri,Episodes_scaled,Score-10_scaled,Score-9_scaled,Score-8_scaled,Score-7_scaled,Score-6_scaled,Score-5_scaled,Score-4_scaled,Score-3_scaled,Score-2_scaled,Score-1_scaled
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",Cowboy Bebop,カウボーイビバップ,TV,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,Bandai Visual,"Funimation, Bandai Entertainment",Sunrise,Original,24 min. per ep.,R - 17+ (violence & profanity),28.0,39,1251960,61971,105808,718161,71513,26678,329800,"Bartender B, Bonnaro, Stella, Long, Ping, Ein,...","Alonso, Tasio, Satou, Dai, Stanzione, Carol, N...","In the year 2071, humanity has colonized sever...",1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0.305709,13.267331,8.959845,6.155067,4.086441,2.897612,2.430365,1.561507,1.304718,1.067769,2.159517
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space",Cowboy Bebop:The Movie,カウボーイビバップ 天国の扉,Movie,"Sep 1, 2001",Unknown,"Sunrise, Bandai Visual",Sony Pictures Entertainment,Bones,Original,1 hr. 55 min.,R - 17+ (violence & profanity),159.0,518,273145,1174,4143,208333,1935,770,57964,"Volaju, Vincent, Robber, Rather, Mark, Spy C, ...","Shibata, Hidekatsu, Isobe, Tsutomu, Kramer, St...","other day, another bounty—such is the life of ...",1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,-0.222305,1.609757,2.277185,2.168147,1.305439,0.594394,0.272312,0.070605,0.009380,-0.033806,0.379343
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen",Trigun,トライガン,TV,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,Victor Entertainment,"Funimation, Geneon Entertainment USA",Madhouse,Manga,24 min. per ep.,PG-13 - Teens 13 or older,266.0,201,558913,12944,29113,343492,25465,13925,146918,"Sandy, Bluesummers, Legato, Schezar, Cliff, Bo...","Pappas, Constantin, Stanzione, Carol, Motoi, E...","Vash the Stampede is the man with a $$60,000,0...",1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0.305709,2.791515,3.606930,3.946871,3.182885,2.075554,1.488769,0.864380,0.514516,0.326995,0.607608
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",Witch Hunter Robin,Witch Hunter ROBIN (ウイッチハンターロビン),TV,"Jul 2, 2002 to Dec 24, 2002",Summer 2002,"TV Tokyo, Bandai Visual, Dentsu, Victor Entert...","Funimation, Bandai Entertainment",Sunrise,Original,25 min. per ep.,PG-13 - Teens 13 or older,2481.0,1467,94683,587,4300,46165,5121,5378,33719,"Amon, Kousaka, Shintarou, Sena, Robin, Sakaki,...","Meyer, Dirk, Cericola, Dania, Takahashi, Kumik...",ches are individuals with special powers like ...,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0.305709,-0.021320,0.045275,0.256397,0.533864,0.579538,0.592626,0.359979,0.159895,0.062059,0.011746
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",Beet the Vandel Buster,冒険王ビィト,TV,"Sep 30, 2004 to Sep 29, 2005",Fall 2004,"TV Tokyo, Dentsu",Unknown,Toei Animation,Manga,23 min. per ep.,PG - Children,3710.0,4369,13224,18,642,7314,766,1108,3394,"Kissu, Slade, Beltoze, Grunide, Poala, Beet, S...","Clinkenbeard, Colleen, Hisakawa, Aya, Yamamuro...",It is the dark century and the people are suff...,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0.854843,-0.130797,-0.169747,-0.175018,-0.160020,-0.138680,-0.109424,-0.107823,-

## Create anime_popularity_score

In [ ]:
anime_features = anime_complete[['Members', 'Favorites', 'Ranked', 'Popularity']].copy()

anime_features = anime_features.fillna(anime_features.median())

print("Check to make sure no invalids:")
print(np.isinf(anime_features).sum())
print(np.isnan(anime_features).sum())

Check to make sure no invalids:
Members       0
Favorites     0
Ranked        0
Popularity    0
dtype: int64
Members       0
Favorites     0
Ranked        0
Popularity    0
dtype: int64


In [ ]:
anime_features['Ranked'] = anime_features['Ranked'].max() - anime_features['Ranked'] # Lower ranked better
anime_features['Popularity'] = anime_features['Popularity'].max() - anime_features['Popularity'] # Lower value for 'Popularity' better

scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(anime_features)

features_scaled = pd.DataFrame(features_scaled, columns=anime_features.columns)

anime_complete['anime_popularity_score'] = (
    0.5 * features_scaled['Members'] +
    0.2 * features_scaled['Favorites'] +
    0.15 * features_scaled['Ranked'] +
    0.15 * features_scaled['Popularity']
)

print("anime_complete with popularity score:")
anime_complete.head()

anime_complete with popularity score:


,anime_id,Name,Score,Genres,English name,Japanese name,Type,Aired,Premiered,Producers,Licensors,Studios,Source,Duration,Rating,Ranked,Popularity,Members,Favorites,Watching,Completed,On-Hold,Dropped,Plan to Watch,Characters,Staff,synopsis,Action,Adventure,Cars,Comedy,Dementia,Demons,Drama,Ecchi,Fantasy,Game,Harem,Hentai,Historical,Horror,Josei,Kids,Magic,Martial Arts,Mecha,Military,Music,Mystery,Parody,Police,Psychological,Romance,Samurai,School,Sci-Fi,Seinen,Shoujo,Shoujo Ai,Shounen,Shounen Ai,Slice of Life,Space,Sports,Super Power,Supernatural,Thriller,Unknown,Vampire,Yaoi,Yuri,Episodes_scaled,Score-10_scaled,Score-9_scaled,Score-8_scaled,Score-7_scaled,Score-6_scaled,Score-5_scaled,Score-4_scaled,Score-3_scaled,Score-2_scaled,Score-1_scaled,anime_popularity_score
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",Cowboy Bebop,カウボーイビバップ,TV,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,Bandai Visual,"Funimation, Bandai Entertainment",Sunrise,Original,24 min. per ep.,R - 17+ (violence & profanity),28.0,39,1251960,61971,105808,718161,71513,26678,329800,"Bartender B, Bonnaro, Stella, Long, Ping, Ein,...","Alonso, Tasio, Satou, Dai, Stanzione, Carol, N...","In the year 2071, humanity has colonized sever...",1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0.305709,13.267331,8.959845,6.155067,4.086441,2.897612,2.430365,1.561507,1.304718,1.067769,2.159517,0.608525
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space",Cowboy Bebop:The Movie,カウボーイビバップ 天国の扉,Movie,"Sep 1, 2001",Unknown,"Sunrise, Bandai Visual",Sony Pictures Entertainment,Bones,Original,1 hr. 55 min.,R - 17+ (violence & profanity),159.0,518,273145,1174,4143,208333,1935,770,57964,"Volaju, Vincent, Robber, Rather, Mark, Spy C, ...","Shibata, Hidekatsu, Isobe, Tsutomu, Kramer, St...","other day, another bounty—such is the life of ...",1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,-0.222305,1.609757,2.277185,2.168147,1.305439,0.594394,0.272312,0.070605,0.009380,-0.033806,0.379343,0.348081
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen",Trigun,トライガン,TV,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,Victor Entertainment,"Funimation, Geneon Entertainment USA",Madhouse,Manga,24 min. per ep.,PG-13 - Teens 13 or older,266.0,201,558913,12944,29113,343492,25465,13925,146918,"Sandy, Bluesummers, Legato, Schezar, Cliff, Bo...","Pappas, Constantin, Stanzione, Carol, Motoi, E...","Vash the Stampede is the man with a $$60,000,0...",1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0.305709,2.791515,3.606930,3.946871,3.182885,2.075554,1.488769,0.864380,0.514516,0.326995,0.607608,0.417748
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",Witch Hunter Robin,Witch Hunter ROBIN (ウイッチハンターロビン),TV,"Jul 2, 2002 to Dec 24, 2002",Summer 2002,"TV Tokyo, Bandai Visual, Dentsu, Victor Entert...","Funimation, Bandai Entertainment",Sunrise,Original,25 min. per ep.,PG-13 - Teens 13 or older,2481.0,1467,94683,587,4300,46165,5121,5378,33719,"Amon, Kousaka, Shintarou, Sena, Robin, Sakaki,...","Meyer, Dirk, Cericola, Dania, Takahashi, Kumik...",ches are individuals with special powers like ...,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0.305709,-0.021320,0.045275,0.256397,0.533864,0.579538,0.592626,0.359979,0.159895,0.062059,0.011746,0.282808
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",Beet the Vandel Buster,冒険王ビィト,TV,"Sep 30, 2004 to Sep 29, 2005",Fall 2004,"TV Tokyo, Dentsu",Unknown,Toei Animation,Manga,23 min. per ep.,PG - Children,3710.0,4369,13224,18,642,7314,766,1108,3394,"Kissu, Slade, Beltoze, Grunide, Poala, Beet, S...","Clinkenbeard, Colleen, Hisakawa, Aya, Yamamuro...",It is the dark century and the people are suff...,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0.854843,-0.130797,-0

## Save the new anime_complete

In [ ]:
# Save the file as .csv
anime_complete.to_csv("anime_complete_encoded.csv")

In [ ]:
# Save the file as .npy

anime_complete_array = anime_complete.to_numpy()
np.save("anime_complete_encoded.npy", anime_complete_array)
print("anime_completed_encoded saved to a npy file.")

anime_completed_encoded saved to a npy file.
